In [2]:
import os
import certifi
import requests

from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub


In [3]:
from langchain.agents import create_react_agent,AgentExecutor

In [4]:
os.environ["SSL_CERT_FILE"] = certifi.where()

load_dotenv(override=True)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [5]:
search_tool = TavilySearchResults(api_key=TAVILY_API_KEY,max_results=3)

In [11]:
@tool
def get_weather_data(city: str) -> str:
    """Fetch current weather data for a city from weatherstak.com"""    
    if not WEATHERSTACK_API_KEY:
        raise ValueError("WEATHERSTACK_API_KEY is not set in the environment variables.")

    url = f"http://api.weatherstack.com/current?access_key={WEATHERSTACK_API_KEY}&query={city}"
    response = requests.get(url)
    data = response.json()

    if "error" in data:
        return f"Error fetching weather data: {data['error']['info']}"

    if "current" not in data:
        return "Error: No current weather data found."
    current_weather = data.get("current", {})
    temperature = current_weather.get("temperature")
    humidity = current_weather.get("humidity")
    weather_descriptions = current_weather.get("weather_descriptions", [])
    description = weather_descriptions[0] if weather_descriptions else "No description available"

    return {
        "city": city,
        "temperature": temperature,
        "humidity": humidity,
        "description": description
    }

In [7]:
llm = ChatOpenAI(
    model_name="gpt-4", 
    temperature=0, openai_api_key=OPENAI_API_KEY)

In [8]:
prompt = hub.pull("hwchase17/react")

/home/pranjal/projects/agentic-ai/.env_agent1/lib/python3.12/site-packages/langchain/hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [13]:
tools = [search_tool,get_weather_data]
# tools = []

In [14]:
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt,
    
)

In [15]:
agent_exe = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True)

In [16]:
response = agent_exe.invoke({"input":"tell me the temprate in ghansoli navi mumbai"})



> Entering new AgentExecutor chain...
The question is asking for the current temperature in a specific location. I can use the get_weather_data function to get this information.
Action: get_weather_data
Action Input: Ghansoli Navi Mumbai{'city': 'Ghansoli Navi Mumbai', 'temperature': 28, 'humidity': 72, 'description': 'Smoky haze'}I now have the temperature data for Ghansoli Navi Mumbai.
Final Answer: The current temperature in Ghansoli Navi Mumbai is 28 degrees Celsius.

> Finished chain.


In [47]:
response

{'input': 'tell me the latest news about Russia',
 'output': "I'm sorry, I can't provide the latest news about Russia at the moment."}